# 04 — Feature Engineering

**PACE Phase**: Analyze
**Objective**: enrich the cleaned dataset with derived columns that will serve
as the basis for answering the research questions. The features created in this
notebook are saved to the final parquet file, so they are immediately
available in all subsequent EDA notebooks.

**Input**: `data/processed/crimes_clean.parquet`
**Output**: `data/processed/crimes_features.parquet`

In [1]:
import pandas as pd # Import the Pandas framework for DataFrame manipulation
import numpy as np  # Import the Numpy framework to handle numeric operations

pd.set_option('display.max_rows', None)         # Pandas setting to show all rows with inspection commands
pd.set_option('display.max_columns', None)      # Pandas setting to show all columns with inspection commands
pd.set_option('display.max_info_columns', 200)  # Pandas setting to show info for 200 columns with the '.info()' command

In [2]:
df = pd.read_parquet('../../data/processed/crimes_clean.parquet') # Create the variable 'df' containing the cleaned
                                                                   # dataset produced by 02_cleaning.ipynb
df.shape # Display the number of rows and columns

(3079424, 24)

## 2. Temporal features

We extract from `DATE OCC` the temporal components that will be needed in
almost all analyses: yearly trends (Block 1), seasonal patterns (Q1.4),
distribution by day of the week (Q1.5).

In [3]:
df['year'] = df['DATE OCC'].dt.year                  # '.dt.year' extracts the year from 'DATE OCC', for yearly trends
df['month'] = df['DATE OCC'].dt.month                 # '.dt.month' extracts the month, for seasonal patterns
df['day_of_week'] = df['DATE OCC'].dt.day_name()      # '.dt.day_name()' extracts the weekday name, for day-of-week distribution

In [4]:
print(df.head(5)) # Display the first 5 rows of the DataFrame to check the new temporal columns

       DR_NO  Date Rptd   DATE OCC  AREA  AREA NAME  Rpt Dist No  Part 1-2  \
0    1307355 2010-02-20 2010-02-20    13     Newton         1385         2   
1   11401303 2010-09-13 2010-09-12    14    Pacific         1485         2   
2   70309629 2010-08-09 2010-08-09    13     Newton         1324         2   
3   90631215 2010-01-05 2010-01-05     6  Hollywood          646         2   
4  100100501 2010-01-03 2010-01-02     1    Central          176         1   

   Crm Cd                                        Crm Cd Desc         Mocodes  \
0     900                           VIOLATION OF COURT ORDER  0913 1814 2000   
1     740  VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...            0329   
2     946                          OTHER MISCELLANEOUS CRIME            0344   
3     900                           VIOLATION OF COURT ORDER  1100 0400 1402   
4     122                                    RAPE, ATTEMPTED            0400   

   Vict Age Vict Sex Vict Descent  Premis Cd      

## 3. Time slot

We group `hour_occ` into 4-hour slots to simplify the temporal
distribution analyses. The slots reflect the rhythms of the day:
night, early morning, morning, afternoon, evening, late night.

In [5]:
df['hour_bins'] = pd.cut(
    df['hour_occ'],
    bins=[-1, 3, 7, 11, 15, 19, 23], # Bin edges splitting the 0-23 hour range into six 4-hour slots
    labels=['Night', 'Early Morning', 'Morning', 'Afternoon', 'Evening', 'Late Night'] # Label for each bin, in
                                                                                       # chronological order
    )
# 'pd.cut' assigns each 'hour_occ' value to the slot whose interval contains it

In [6]:
print(df['hour_bins'].value_counts()) # Display the count of records in each time slot

hour_bins
Evening          690927
Afternoon        678361
Late Night       630808
Morning          498862
Night            351450
Early Morning    229016
Name: count, dtype: int64


## 4. Age groups

We group `Vict Age` into brackets that reflect the categories relevant
to the research questions: children, adolescents, young adults, adults,
seniors. These brackets will be used in particular for Block 2 (victim
profile) and Block 3 (domestic abuse and minor safety, with a focus
on the 0-12, 13-17, 18+ brackets).

In [7]:
df['age_group'] = pd.cut(
    df['Vict Age'],
    bins=[-1, 12, 17, 34, 64, 120], # Bin edges for the 0-12, 13-17, 18-34, 35-64, 65+ brackets
    labels=['Child', 'Adolescent', 'Young Adult', 'Adult', 'Senior'] # Label for each bracket
)
# Records with 'Vict Age' still null (unknown-victim crimes) get NaN here too, and are excluded
# from age-based analyses, consistent with how the sentinel values were handled in cleaning

print(df['age_group'].value_counts(dropna=False)) # Display the count per bracket, including the NaN bucket

age_group
Adult          1158297
Young Adult     991025
NaN             632392
Senior          169266
Adolescent       84156
Child            44288
Name: count, dtype: int64


## 5. Crime macro-category

We classify crimes into two macro-categories — "Person" (against the person)
and "Property" (against property) — needed for the Block 1 analyses
(Q1.2: separate person vs. property trends). The classification is based on
two curated lists of `Crm Cd Desc` values (`person_crimes`, `property_crimes`),
built by manually reviewing all distinct crime descriptions in the dataset.
Descriptions not found in either list are classified as "other".

In [8]:
df['Crm Cd Desc'].unique() # Display all distinct crime descriptions, reviewed manually to build the
                           # 'person_crimes' / 'property_crimes' lists in the next cell

array(['VIOLATION OF COURT ORDER',
       'VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)',
       'OTHER MISCELLANEOUS CRIME', 'RAPE, ATTEMPTED',
       'SHOPLIFTING - PETTY THEFT ($950 & UNDER)',
       'BURGLARY FROM VEHICLE',
       'ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT',
       'THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LIVESTK,PROD',
       'BATTERY - SIMPLE ASSAULT', 'ROBBERY', 'BOMB SCARE',
       'CHILD NEGLECT (SEE 300 W.I.C.)',
       'INTIMATE PARTNER - AGGRAVATED ASSAULT',
       'INTIMATE PARTNER - SIMPLE ASSAULT',
       'THEFT PLAIN - PETTY ($950 & UNDER)',
       'CRIMINAL THREATS - NO WEAPON DISPLAYED', 'ATTEMPTED ROBBERY',
       'VANDALISM - MISDEAMEANOR ($399 OR UNDER)', 'BURGLARY', 'ARSON',
       'RAPE, FORCIBLE', 'BRANDISH WEAPON',
       'THROWING OBJECT AT MOVING VEHICLE',
       'SHOPLIFTING-GRAND THEFT ($950.01 & OVER)',
       'CHILD ABUSE (PHYSICAL) - SIMPLE ASSAULT',
       'SHOTS FIRED AT INHABITED DWELLING', 'VEHICLE - STOLEN',
    

In [9]:
# List of 'Crm Cd Desc' values manually classified as crimes against the person
person_crimes = [
    'RAPE, ATTEMPTED',
    'ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT',
    'BATTERY - SIMPLE ASSAULT',
    'ROBBERY',
    'CHILD NEGLECT (SEE 300 W.I.C.)',
    'INTIMATE PARTNER - AGGRAVATED ASSAULT',
    'INTIMATE PARTNER - SIMPLE ASSAULT',
    'CRIMINAL THREATS - NO WEAPON DISPLAYED',
    'ATTEMPTED ROBBERY',
    'RAPE, FORCIBLE',
    'BRANDISH WEAPON',
    'CHILD ABUSE (PHYSICAL) - SIMPLE ASSAULT',
    'KIDNAPPING - GRAND ATTEMPT',
    'CRIMINAL HOMICIDE',
    'THEFT, PERSON',
    'KIDNAPPING',
    'BATTERY WITH SEXUAL CONTACT',
    'BATTERY POLICE (SIMPLE)',
    'CHILD ABUSE (PHYSICAL) - AGGRAVATED ASSAULT',
    'OTHER ASSAULT',
    'VIOLATION OF RESTRAINING ORDER',
    'VIOLATION OF COURT ORDER',
    'LEWD CONDUCT',
    'CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 YRS OLDER)',
    'ASSAULT WITH DEADLY WEAPON ON POLICE OFFICER',
    'CHILD ANNOYING (17YRS & UNDER)',
    'SODOMY/SEXUAL CONTACT B/W PENIS OF ONE PERS TO ANUS OTH',
    'ORAL COPULATION',
    'LETTERS, LEWD  -  TELEPHONE CALLS, LEWD',
    'PEEPING TOM',
    'INDECENT EXPOSURE',
    'STALKING',
    'SEXUAL PENETRATION W/FOREIGN OBJECT',
    'THREATENING PHONE CALLS/LETTERS',
    'SEX,UNLAWFUL(INC MUTUAL CONSENT, PENETRATION W/ FRGN OBJ',
    'EXTORTION',
    'PICKPOCKET',
    'PURSE SNATCHING',
    'FALSE IMPRISONMENT',
    'DISCHARGE FIREARMS/SHOTS FIRED',
    'THEFT FROM PERSON - ATTEMPT',
    'PANDERING',
    'PIMPING',
    'DRUNK ROLL - ATTEMPT',
    'RESISTING ARREST',
    'CHILD STEALING',
    'DRUNK ROLL',
    'BATTERY ON A FIREFIGHTER',
    'LYNCHING',
    'PURSE SNATCHING - ATTEMPT',
    'SHOTS FIRED AT INHABITED DWELLING',
    'SHOTS FIRED AT MOVING VEHICLE, TRAIN OR AIRCRAFT',
    'CONTRIBUTING',
    'DRUGS, TO A MINOR',
    'INCITING A RIOT',
    'PICKPOCKET, ATTEMPT',
    'LYNCHING - ATTEMPTED',
    'CHILD ABANDONMENT',
    'LEWD/LASCIVIOUS ACTS WITH CHILD',
    'BEASTIALITY, CRIME AGAINST NATURE SEXUAL ASSLT WITH ANIM',
    'HUMAN TRAFFICKING - COMMERCIAL SEX ACTS',
    'MANSLAUGHTER, NEGLIGENT',
    'HUMAN TRAFFICKING - INVOLUNTARY SERVITUDE',
    'CHILD PORNOGRAPHY',
    'ABORTION/ILLEGAL',
    'INCEST (SEXUAL ACTS BETWEEN BLOOD RELATIVES)',
    'VIOLATION OF TEMPORARY RESTRAINING ORDER',
    'TILL TAP - ATTEMPT',
    'TILL TAP - GRAND THEFT ($950.01 & OVER)',
    'TILL TAP - PETTY ($950 & UNDER)'
]

# List of 'Crm Cd Desc' values manually classified as crimes against property
property_crimes = [
    'VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)',
    'SHOPLIFTING - PETTY THEFT ($950 & UNDER)',
    'BURGLARY FROM VEHICLE',
    'THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LIVESTK,PROD',
    'THEFT PLAIN - PETTY ($950 & UNDER)',
    'VANDALISM - MISDEAMEANOR ($399 OR UNDER)',
    'BURGLARY',
    'ARSON',
    'SHOPLIFTING-GRAND THEFT ($950.01 & OVER)',
    'VEHICLE - STOLEN',
    'THEFT PLAIN - ATTEMPT',
    'TRESPASSING',
    'VEHICLE - ATTEMPT STOLEN',
    'DOCUMENT FORGERY / STOLEN FELONY',
    'EMBEZZLEMENT, GRAND THEFT ($950.01 & OVER)',
    'THEFT OF IDENTITY',
    'CRUELTY TO ANIMALS',
    'THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER)',
    'BIKE - STOLEN',
    'BURGLARY FROM VEHICLE, ATTEMPTED',
    'BURGLARY, ATTEMPTED',
    'BUNCO, PETTY THEFT',
    'BUNCO, GRAND THEFT',
    'EMBEZZLEMENT, PETTY THEFT ($950 & UNDER)',
    'WEAPONS POSSESSION/BOMBING',
    'COUNTERFEIT',
    'CREDIT CARDS, FRAUD USE ($950.01 & OVER)',
    'UNAUTHORIZED COMPUTER ACCESS',
    'SHOPLIFTING - ATTEMPT',
    'DISHONEST EMPLOYEE - GRAND THEFT',
    'CREDIT CARDS, FRAUD USE ($950 & UNDER',
    'DOCUMENT WORTHLESS ($200.01 & OVER)',
    'CONSPIRACY',
    'THEFT FROM MOTOR VEHICLE - ATTEMPT',
    'ILLEGAL DUMPING',
    'THEFT, COIN MACHINE - PETTY ($950 & UNDER)',
    'GRAND THEFT / INSURANCE FRAUD',
    'BUNCO, ATTEMPT',
    'THEFT, COIN MACHINE - GRAND ($950.01 & OVER)',
    'BOAT - STOLEN',
    'DRIVING WITHOUT OWNER CONSENT (DWOC)',
    'THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND OVER)',
    'DEFRAUDING INNKEEPER/THEFT OF SERVICES, $950 & UNDER',
    'DEFRAUDING INNKEEPER/THEFT OF SERVICES, OVER $950.01',
    'BRIBERY',
    'DISHONEST EMPLOYEE - PETTY THEFT',
    'TELEPHONE PROPERTY - DAMAGE',
    'DOCUMENT WORTHLESS ($200 & UNDER)',
    'GRAND THEFT / AUTO REPAIR',
    'THEFT, COIN MACHINE - ATTEMPT',
    'PETTY THEFT - AUTO REPAIR',
    'BIKE - ATTEMPTED STOLEN',
    'DISHONEST EMPLOYEE ATTEMPTED THEFT',
    'TRAIN WRECKING',
    'VEHICLE, STOLEN - OTHER (MOTORIZED SCOOTERS, BIKES, ETC)',
    'THROWING OBJECT AT MOVING VEHICLE'
]

cond1 = df['Crm Cd Desc'].isin(property_crimes) # Boolean mask: True where the crime description is in 'property_crimes'
cond2 = df['Crm Cd Desc'].isin(person_crimes)   # Boolean mask: True where the crime description is in 'person_crimes'

df['crime_category'] = np.select(
[cond1, cond2],           # Conditions, evaluated in order
['property', 'person'],   # Value assigned when the matching condition is True
default = 'other'         # Value assigned to descriptions found in neither list
)

In [10]:
df['crime_category'].value_counts() # Display the count of records in each macro-category

crime_category
property    1938752
person      1093024
other         47648
Name: count, dtype: int64

## 6. Report delay

We calculate the difference in days between the report date (`Date Rptd`)
and the date the crime occurred (`DATE OCC`). This feature measures
the reporting delay and will be used for Block 5 (Q5.2). Different crimes
have very different delays: a homicide is reported immediately, a sexual
assault often years later.

In [11]:
df['report_delay']=(df['Date Rptd']-df['DATE OCC']).dt.days # Subtract 'DATE OCC' from 'Date Rptd' and take
                                                             # '.dt.days' to get the delay in whole days
df['report_delay'].describe().round(2) # Display descriptive statistics of the new column

count    3079424.00
mean          21.06
std          156.57
min            0.00
25%            0.00
50%            1.00
75%            2.00
max         5407.00
Name: report_delay, dtype: float64

## 7. Domestic crime flag

We create a boolean column `is_domestic` to quickly identify
domestic crimes. The classification is based on keywords in
`Crm Cd Desc` that indicate domestic violence or child abuse.
It will serve as a quick filter for all of Block 3 and potentially
for cross-cutting analyses in the other blocks.

In [12]:
domestic_keywords = 'INTIMATE PARTNER|CHILD ABUSE|CHILD NEGLECT|CHILD ABANDONMENT|CHILD STEALING|CHILD ANNOYING|CRM AGNST CHLD|INCEST'
# Pipe-separated keywords identifying domestic violence and child-abuse related crime descriptions

df['is_domestic'] = df['Crm Cd Desc'].str.contains(
    pat = domestic_keywords, # '.str.contains' flags rows whose description matches any of the keywords (regex OR)
    case = False             # Case-insensitive match
)

df['is_domestic'].value_counts() # Display the count of domestic vs. non-domestic records

is_domestic
False    2853486
True      225938
Name: count, dtype: int64

## 8. Final check and saving

Summary of the features created and saving of the enriched dataset.

In [13]:
print(f'The number of rows and columns of the DataFrame is: {df.shape}') # Display the DataFrame's shape
print('DataFrame information:')                                          # Display the given string
print(df.info(show_counts=True))                                         # Display dtypes and non-null counts
                                                                          # for every column, including the new features

The number of rows and columns of the DataFrame is: (3079424, 32)
DataFrame information:
<class 'pandas.core.frame.DataFrame'>
Index: 3079424 entries, 0 to 3138030
Data columns (total 32 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   DR_NO           3079424 non-null  int64         
 1   Date Rptd       3079424 non-null  datetime64[ns]
 2   DATE OCC        3079424 non-null  datetime64[ns]
 3   AREA            3079424 non-null  int64         
 4   AREA NAME       3079424 non-null  object        
 5   Rpt Dist No     3079424 non-null  int64         
 6   Part 1-2        3079424 non-null  int64         
 7   Crm Cd          3079424 non-null  int64         
 8   Crm Cd Desc     3079424 non-null  object        
 9   Mocodes         2704233 non-null  object        
 10  Vict Age        2447032 non-null  float64       
 11  Vict Sex        3079424 non-null  object        
 12  Vict Descent    3079424 non-null  object  

In [14]:
df.to_parquet('../../data/processed/crimes_features.parquet') # '.to_parquet' saves the enriched DataFrame,
                                                                # the input for all subsequent EDA notebooks
print(f"✅ Dataset saved to data/processed/crimes_features.parquet") # Display confirmation message
print(f"   Rows: {len(df):,}")           # Display the final row count
print(f"   Columns: {df.shape[1]}")      # Display the final column count

✅ Dataset saved to data/processed/crimes_features.parquet
   Rows: 3,079,424
   Columns: 32
